In [1]:
# prime_fock_ablation.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal, Optional, Sequence

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    roc_auc_score,
    log_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

# Adjust this import to your package name if needed.
# Example:
from prime_fock_nd import (
     PrimeFockNDEncoder,
     ArithmeticGraphBuilder,
     StabilityFilter,
     GenericFeatureEmitter,
)
from .encoder import PrimeFockNDEncoder
from .graph import ArithmeticGraphBuilder
from .stability import StabilityFilter
from .features import GenericFeatureEmitter


TaskKind = Literal["regression", "classification"]
WeightScheme = Literal[
    "log_prime_weights",
    "uniform_weights",
    "random_monotone_weights",
    "learned_weights",
    "shuffled_mode_weights",
]


@dataclass
class AblationResult:
    scheme: str
    seed: int
    metrics: dict
    feature_columns: list[str]
    weights: np.ndarray


def _soft_normalize_weights(weights: np.ndarray, reference: np.ndarray) -> np.ndarray:
    """
    Normalize weights to have the same mean scale as log-prime weights.
    This keeps ablations comparable: the graph does not become stronger/weaker
    just because the absolute weight scale changed.
    """
    weights = np.asarray(weights, dtype=float)
    weights = np.maximum(weights, 1e-12)

    ref_mean = float(np.mean(reference))
    w_mean = float(np.mean(weights))

    if w_mean <= 1e-12:
        return np.ones_like(weights) * ref_mean

    return weights * (ref_mean / w_mean)


def _flatten_encoded_modes(encoded) -> pd.DataFrame:
    """
    Convert encoded.exponents with shape (S, I, P) into row-level dataframe:

        columns: slice, item, mode_0, ..., mode_{P-1}

    This is useful for learning mode weights from training data.
    """
    x = np.asarray(encoded.exponents, dtype=float)
    S, I, P = x.shape

    rows = []
    for s_idx, s_key in enumerate(encoded.slices):
        for i_idx, item in enumerate(encoded.items):
            row = {
                "slice": s_key,
                "item": item,
            }
            for p in range(P):
                row[f"mode_{p}"] = float(x[s_idx, i_idx, p])
            rows.append(row)

    return pd.DataFrame(rows)


def _make_key_df(df: pd.DataFrame, slice_cols: Sequence[str], item_col: str) -> pd.DataFrame:
    """
    Build merge keys compatible with the feature emitter output.

    The library stores slice labels as tuples, even for a single slice column.
    """
    out = pd.DataFrame(index=df.index)

    if len(slice_cols) == 0:
        out["slice"] = [("__all__",)] * len(df)
    else:
        out["slice"] = [tuple(row) for row in df[list(slice_cols)].to_numpy()]

    out["item"] = df[item_col].to_numpy()
    return out


def _learn_mode_weights(
    encoded,
    original_df: pd.DataFrame,
    y_col: str,
    slice_cols: Sequence[str],
    item_col: str,
    task: TaskKind,
    reference_log_weights: np.ndarray,
) -> np.ndarray:
    """
    Supervised learned mode weights.

    This learns one nonnegative importance weight per Fock mode using only
    the provided dataframe. In the ablation runner below, this is called only
    on the training split, so it avoids test leakage.

    Uses mutual information because:
    - it is nonparametric,
    - handles nonlinear relevance,
    - returns nonnegative scores,
    - works for regression and classification.
    """
    mode_df = _flatten_encoded_modes(encoded)
    key_df = _make_key_df(original_df, slice_cols, item_col)
    target_df = key_df.copy()
    target_df[y_col] = original_df[y_col].to_numpy()

    merged = mode_df.merge(target_df, on=["slice", "item"], how="inner")

    mode_cols = [c for c in merged.columns if c.startswith("mode_")]
    X_modes = merged[mode_cols].to_numpy(dtype=float)
    y = merged[y_col].to_numpy()

    if X_modes.shape[0] < 5:
        return reference_log_weights.copy()

    # If all target values are identical, no supervised weighting is meaningful.
    if len(np.unique(y)) <= 1:
        return reference_log_weights.copy()

    if task == "regression":
        scores = mutual_info_regression(X_modes, y, random_state=0)
    else:
        scores = mutual_info_classif(X_modes, y, random_state=0)

    scores = np.asarray(scores, dtype=float)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

    # Stabilize: do not allow totally zero graph unless MI really has signal.
    if np.all(scores <= 1e-12):
        return np.ones_like(reference_log_weights) * np.mean(reference_log_weights)

    # Add tiny floor so unused-but-possible modes are not completely erased.
    scores = scores + 1e-6

    return _soft_normalize_weights(scores, reference_log_weights)


def _make_weights(
    scheme: WeightScheme,
    encoded_train,
    train_df: pd.DataFrame,
    y_col: str,
    slice_cols: Sequence[str],
    item_col: str,
    task: TaskKind,
    seed: int,
) -> np.ndarray | None:
    """
    Return mode_weights for graph builder.

    None means use the library default: log(primes).
    """
    rng = np.random.default_rng(seed)

    P = encoded_train.exponents.shape[2]
    log_prime_weights = np.log(encoded_train.primes.astype(float))

    if scheme == "log_prime_weights":
        return None

    if scheme == "uniform_weights":
        return np.ones(P, dtype=float) * float(np.mean(log_prime_weights))

    if scheme == "random_monotone_weights":
        # Random increasing sequence, normalized to log-prime average scale.
        raw = np.cumsum(rng.gamma(shape=2.0, scale=1.0, size=P))
        return _soft_normalize_weights(raw, log_prime_weights)

    if scheme == "learned_weights":
        return _learn_mode_weights(
            encoded=encoded_train,
            original_df=train_df,
            y_col=y_col,
            slice_cols=slice_cols,
            item_col=item_col,
            task=task,
            reference_log_weights=log_prime_weights,
        )

    if scheme == "shuffled_mode_weights":
        # Preserve the exact log-prime weight distribution, but break the
        # association between mode identity and prime weight.
        shuffled = log_prime_weights.copy()
        rng.shuffle(shuffled)
        return shuffled

    raise ValueError(f"Unknown scheme: {scheme}")


def _build_features(
    df: pd.DataFrame,
    encoder: PrimeFockNDEncoder,
    mode_weights: Optional[np.ndarray],
    graph_kernel: str = "log_gcd",
) -> pd.DataFrame:
    """
    Transform dataframe into model-ready feature rows using:

        encoder -> ArithmeticGraphBuilder -> StabilityFilter -> GenericFeatureEmitter
    """
    encoded = encoder.transform(df)

    graph = ArithmeticGraphBuilder(
        mode_weights=mode_weights,
        kernel=graph_kernel,
    ).build(encoded)

    stability = StabilityFilter().fit_transform(graph)

    features = GenericFeatureEmitter().transform(
        encoded=encoded,
        graph=graph,
        stability=stability,
    )

    return features


def _prepare_xy(
    features: pd.DataFrame,
    source_df: pd.DataFrame,
    y_col: str,
    slice_cols: Sequence[str],
    item_col: str,
) -> tuple[pd.DataFrame, np.ndarray]:
    """
    Join emitted graph features back to the target column.
    """
    key_df = _make_key_df(source_df, slice_cols, item_col)
    target_df = key_df.copy()
    target_df[y_col] = source_df[y_col].to_numpy()

    merged = features.merge(target_df, on=["slice", "item"], how="inner")

    drop_cols = ["slice", "item", y_col]
    X = merged.drop(columns=drop_cols)

    # Keep only numeric model inputs.
    X = X.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    y = merged[y_col].to_numpy()

    return X, y


def _default_model(task: TaskKind, seed: int):
    if task == "regression":
        return RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=3,
            random_state=seed,
            n_jobs=-1,
        )

    return RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=3,
        random_state=seed,
        n_jobs=-1,
        class_weight="balanced",
    )


def _score_model(model, X_test: pd.DataFrame, y_test: np.ndarray, task: TaskKind) -> dict:
    pred = model.predict(X_test)

    if task == "regression":
        rmse = mean_squared_error(y_test, pred, squared=False)
        mae = mean_absolute_error(y_test, pred)
        r2 = r2_score(y_test, pred)
        corr = np.corrcoef(y_test, pred)[0, 1] if len(y_test) > 1 else np.nan

        return {
            "r2": float(r2),
            "rmse": float(rmse),
            "mae": float(mae),
            "pearson_corr": float(corr),
        }

    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
    }

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)

        try:
            metrics["log_loss"] = float(log_loss(y_test, proba))
        except Exception:
            pass

        # Binary AUC only.
        if proba.shape[1] == 2 and len(np.unique(y_test)) == 2:
            metrics["roc_auc"] = float(roc_auc_score(y_test, proba[:, 1]))

    return metrics


def run_prime_fock_weight_ablation(
    df: pd.DataFrame,
    config,
    y_col: str,
    task: TaskKind = "regression",
    schemes: Sequence[WeightScheme] = (
        "log_prime_weights",
        "uniform_weights",
        "random_monotone_weights",
        "learned_weights",
        "shuffled_mode_weights",
    ),
    seeds: Sequence[int] = (0, 1, 2, 3, 4),
    test_size: float = 0.25,
    graph_kernel: str = "log_gcd",
    model=None,
    split_by_slice: bool = True,
) -> tuple[pd.DataFrame, list[AblationResult]]:
    """
    Run ablations over mode-weight schemes.

    Parameters
    ----------
    df:
        Original dataframe containing slice/item/spec columns and y_col.

    config:
        PrimeFockNDConfig.

    y_col:
        Target column.

    task:
        "regression" or "classification".

    schemes:
        Weight ablations to test.

    seeds:
        Random seeds.

    test_size:
        Test fraction.

    graph_kernel:
        Usually "log_gcd". You can also test "jaccard_gcd_lcm",
        "arithmetic_distance", etc. Dense builder supports these.

    model:
        Optional sklearn estimator. If None, uses RandomForest.

    split_by_slice:
        If True, train/test split is done by slice keys. This is better for
        temporal/panel data because it reduces leakage across items in the
        same time slice. If False, splits rows randomly.

    Returns
    -------
    summary_df:
        One row per scheme/seed with metrics.

    raw_results:
        Full AblationResult objects, including learned/used weights.
    """
    if y_col not in df.columns:
        raise ValueError(f"Target column {y_col!r} not found in df.")

    item_col = config.item_col
    slice_cols = list(config.slice_cols)

    raw_results: list[AblationResult] = []
    summary_rows = []

    for seed in seeds:
        if split_by_slice and len(slice_cols) > 0:
            unique_slices = (
                df[slice_cols]
                .drop_duplicates()
                .sample(frac=1.0, random_state=seed)
            )

            train_slices, test_slices = train_test_split(
                unique_slices,
                test_size=test_size,
                random_state=seed,
            )

            train_keys = set(map(tuple, train_slices.to_numpy()))
            test_keys = set(map(tuple, test_slices.to_numpy()))

            row_keys = [tuple(row) for row in df[slice_cols].to_numpy()]
            train_mask = np.array([k in train_keys for k in row_keys])
            test_mask = np.array([k in test_keys for k in row_keys])

            train_df = df.loc[train_mask].copy()
            test_df = df.loc[test_mask].copy()

        else:
            train_df, test_df = train_test_split(
                df,
                test_size=test_size,
                random_state=seed,
                stratify=df[y_col] if task == "classification" else None,
            )
            train_df = train_df.copy()
            test_df = test_df.copy()

        # Fit encoder only on training data.
        # Important: learned categories/vocab/bin specs must not see test data.
        encoder = PrimeFockNDEncoder(config).fit(train_df)
        encoded_train = encoder.transform(train_df)

        for scheme in schemes:
            mode_weights = _make_weights(
                scheme=scheme,
                encoded_train=encoded_train,
                train_df=train_df,
                y_col=y_col,
                slice_cols=slice_cols,
                item_col=item_col,
                task=task,
                seed=seed,
            )

            train_features = _build_features(
                df=train_df,
                encoder=encoder,
                mode_weights=mode_weights,
                graph_kernel=graph_kernel,
            )

            test_features = _build_features(
                df=test_df,
                encoder=encoder,
                mode_weights=mode_weights,
                graph_kernel=graph_kernel,
            )

            X_train, y_train = _prepare_xy(
                features=train_features,
                source_df=train_df,
                y_col=y_col,
                slice_cols=slice_cols,
                item_col=item_col,
            )

            X_test, y_test = _prepare_xy(
                features=test_features,
                source_df=test_df,
                y_col=y_col,
                slice_cols=slice_cols,
                item_col=item_col,
            )

            if X_train.empty or X_test.empty:
                raise RuntimeError(
                    f"No usable rows for scheme={scheme}, seed={seed}. "
                    "Check slice/item keys and target alignment."
                )

            estimator = clone(model) if model is not None else _default_model(task, seed)
            estimator.fit(X_train, y_train)

            metrics = _score_model(estimator, X_test, y_test, task)

            if mode_weights is None:
                stored_weights = np.log(encoded_train.primes.astype(float))
            else:
                stored_weights = np.asarray(mode_weights, dtype=float)

            result = AblationResult(
                scheme=scheme,
                seed=seed,
                metrics=metrics,
                feature_columns=list(X_train.columns),
                weights=stored_weights,
            )
            raw_results.append(result)

            row = {
                "scheme": scheme,
                "seed": seed,
                **metrics,
            }
            summary_rows.append(row)

            print(f"[seed={seed}] {scheme}: {metrics}")

    summary_df = pd.DataFrame(summary_rows)

    return summary_df, raw_results


def summarize_ablation(summary_df: pd.DataFrame, task: TaskKind = "regression") -> pd.DataFrame:
    """
    Aggregate ablation results across seeds.
    """
    metric_cols = [
        c for c in summary_df.columns
        if c not in {"scheme", "seed"}
    ]

    grouped = summary_df.groupby("scheme")[metric_cols].agg(["mean", "std", "median"])
    grouped.columns = ["_".join(col).strip() for col in grouped.columns.to_flat_index()]
    grouped = grouped.reset_index()

    if task == "regression":
        sort_col = "r2_mean" if "r2_mean" in grouped.columns else grouped.columns[1]
        grouped = grouped.sort_values(sort_col, ascending=False)
    else:
        if "roc_auc_mean" in grouped.columns:
            grouped = grouped.sort_values("roc_auc_mean", ascending=False)
        elif "accuracy_mean" in grouped.columns:
            grouped = grouped.sort_values("accuracy_mean", ascending=False)

    return grouped

ImportError: DLL load failed while importing lib: The specified procedure could not be found.